# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

./sample_data/mnist_train_small.csv
./sample_data/california_housing_train.csv
./sample_data/california_housing_test.csv
./sample_data/mnist_test.csv


In [3]:
!git clone https://github.com/vgrablonjfooooh-maker/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 112 (delta 30), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 1.85 MiB | 15.75 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [4]:
import os
os.chdir("flyrank-ml-internship")
print(os.getcwd())

/content/flyrank-ml-internship


In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [7]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

print(model)

RandomForestRegressor(random_state=42)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
from sklearn.model_selection import train_test_split

df_model = df.copy()

df_model = df_model.select_dtypes(include=["number"])
df_model = df_model.dropna()

X = df_model.drop(columns=["ctr"])
y = df_model["ctr"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(14333, 29)
(3584, 29)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd

model.fit(X_train, y_train)

pred = model.predict(X_test)

results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "R2"],
    "Value": [
        mean_absolute_error(y_test, pred),
        mean_squared_error(y_test, pred),
        r2_score(y_test, pred)
    ]
})

results

,Metric,Value
0,MAE,0.015586
1,MSE,0.016527
2,R2,0.976225


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
comparison = pd.DataFrame({
    "Actual_CTR": y_test.values,
    "Predicted_CTR": pred
})

comparison["Error"] = abs(
    comparison["Actual_CTR"] - comparison["Predicted_CTR"]
)

comparison.sort_values(
    by="Error",
    ascending=False
).head(10)

,Actual_CTR,Predicted_CTR,Error
147,14.29,18.4809,4.1909
3370,11.11,14.5246,3.4146
3007,9.09,11.2165,2.1265
1778,25.00,23.4274,1.5726
1115,9.52,8.0732,1.4468
1683,8.93,7.5891,1.3409
2748,3.56,2.2201,1.3399
528,3.38,2.1377,1.2423
2462,4.29,3.0835,1.2065
3267,6.25,7.4454,1.1954


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.